# 🔥 Ultra-Ensemble Beast Mode - Local VS Code Version

**Target:** TOP 10-100 (43-52% SMAPE)  
**Runtime:** 12-18 hours on local GPU  
**Strategy:** 3 Vision + 3 Text + 6 ML Models + Meta-Stacking

---

## Before Running:
1. Make sure you have **GPU available** (NVIDIA with CUDA)
2. Install required packages (see cell below)
3. Update **data paths** in Section 2
4. Run cells **one by one** to monitor progress

## Expected Timeline:
- Installation: 5-10 min
- Vision features: 4-6 hours
- Text features: 2-3 hours
- Training: 4-6 hours
- **Total: 12-18 hours**

## Section 1: Installation & Setup

In [ ]:
# Install required packages (run once)
# This will take 5-10 minutes

!pip install -q transformers torch torchvision timm pillow
!pip install -q xgboost lightgbm catboost scikit-learn
!pip install -q pandas numpy requests tqdm scipy

print("✓ All packages installed!")

In [ ]:
# Import libraries and check GPU

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from transformers import (
    BertTokenizer, BertModel,
    RobertaTokenizer, RobertaModel,
    CLIPProcessor, CLIPModel
)
import timm
from PIL import Image
import requests
from io import BytesIO
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from tqdm.notebook import tqdm
import warnings
import gc

warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🔥 Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("   ⚠️ WARNING: No GPU detected! This will be VERY slow on CPU.")
    print("   Consider using Kaggle GPU instead.")

## Section 2: Load Data

**⚠️ UPDATE THESE PATHS TO YOUR LOCAL FILES!**

In [ ]:
# UPDATE THESE PATHS!
TRAIN_PATH = 'dataset/sample_train.csv'  # ← Change this to your train file path
TEST_PATH = 'dataset/sample_test.csv'    # ← Change this to your test file path

# Load data
print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"\n✓ Train shape: {train_df.shape}")
print(f"✓ Test shape: {test_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")
print(f"\n📊 Target statistics:\n{train_df['price'].describe()}")

# Show sample
print(f"\n📝 First few rows:")
train_df.head(3)

## Section 3: Vision Feature Extraction (3 Models)

**This will take 4-6 hours for 150K images.**  
Progress bar will show estimated time remaining.

In [ ]:
# Define vision feature extractor (3 models)

class VisionExtractor:
    def __init__(self, device):
        self.device = device
        
        print("Loading vision models...")
        
        # ResNet50
        print("  [1/3] Loading ResNet50...")
        resnet = models.resnet50(pretrained=True)
        self.resnet = nn.Sequential(*list(resnet.children())[:-1])
        self.resnet.eval().to(device)
        
        # EfficientNet-B4
        print("  [2/3] Loading EfficientNet-B4...")
        self.efficientnet = timm.create_model('efficientnet_b4', pretrained=True, num_classes=0)
        self.efficientnet.eval().to(device)
        
        # Vision Transformer
        print("  [3/3] Loading Vision Transformer...")
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        self.vit.eval().to(device)
        
        self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        
        self.success = 0
        self.failed = 0
        
        print("\n✓ All vision models loaded!")
    
    def download_image(self, url):
        try:
            r = requests.get(url, timeout=5)
            img = Image.open(BytesIO(r.content)).convert('RGB')
            self.success += 1
            return img
        except:
            self.failed += 1
            return Image.new('RGB', (224, 224), color='gray')
    
    def extract(self, image_url):
        img = self.download_image(image_url)
        img_tensor = self.transform(img).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            resnet_feat = self.resnet(img_tensor).squeeze().cpu().numpy()
            eff_feat = self.efficientnet(img_tensor).squeeze().cpu().numpy()
            vit_feat = self.vit(img_tensor).squeeze().cpu().numpy()
        
        return np.concatenate([resnet_feat, eff_feat, vit_feat])  # 4608-dim

vision_extractor = VisionExtractor(device)

In [ ]:
# Extract vision features from TRAIN data
# This will take ~2-3 hours for 75K images

print("🖼️ Extracting vision features from TRAINING data...")
print(f"   Processing {len(train_df)} images...\n")

train_vision_features = []

for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Train Vision"):
    features = vision_extractor.extract(row['image_link'])
    train_vision_features.append(features)
    
    # Periodic cleanup
    if idx % 500 == 0 and idx > 0:
        gc.collect()
        torch.cuda.empty_cache()
        success_rate = vision_extractor.success / (idx + 1) * 100
        print(f"   Progress: {idx}/{len(train_df)} | Success: {success_rate:.1f}%")

train_vision_df = pd.DataFrame(train_vision_features, 
                               columns=[f'vision_{i}' for i in range(4608)])

total = vision_extractor.success + vision_extractor.failed
print(f"\n✓ Train vision features extracted!")
print(f"   Shape: {train_vision_df.shape}")
print(f"   Success rate: {vision_extractor.success/total*100:.1f}%")

In [ ]:
# Extract vision features from TEST data
# This will take ~2-3 hours for 75K images

print("🖼️ Extracting vision features from TEST data...")
print(f"   Processing {len(test_df)} images...\n")

# Reset counters
vision_extractor.success = 0
vision_extractor.failed = 0

test_vision_features = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test Vision"):
    features = vision_extractor.extract(row['image_link'])
    test_vision_features.append(features)
    
    if idx % 500 == 0 and idx > 0:
        gc.collect()
        torch.cuda.empty_cache()
        success_rate = vision_extractor.success / (idx + 1) * 100
        print(f"   Progress: {idx}/{len(test_df)} | Success: {success_rate:.1f}%")

test_vision_df = pd.DataFrame(test_vision_features,
                              columns=[f'vision_{i}' for i in range(4608)])

total = vision_extractor.success + vision_extractor.failed
print(f"\n✓ Test vision features extracted!")
print(f"   Shape: {test_vision_df.shape}")
print(f"   Success rate: {vision_extractor.success/total*100:.1f}%")

# Free memory
del vision_extractor
gc.collect()
torch.cuda.empty_cache()

## Section 4: Text Feature Extraction (3 Models)

**This will take 2-3 hours for 150K texts.**

In [ ]:
# Define text feature extractor (3 models)

class TextExtractor:
    def __init__(self, device):
        self.device = device
        
        print("Loading text models...")
        
        # BERT
        print("  [1/3] Loading BERT...")
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.bert_model = BertModel.from_pretrained('bert-base-uncased')
        self.bert_model.eval().to(device)
        
        # RoBERTa
        print("  [2/3] Loading RoBERTa...")
        self.roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
        self.roberta_model = RobertaModel.from_pretrained('roberta-base')
        self.roberta_model.eval().to(device)
        
        # CLIP
        print("  [3/3] Loading CLIP...")
        self.clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
        self.clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
        self.clip_model.eval().to(device)
        
        print("\n✓ All text models loaded!")
    
    def extract(self, text):
        with torch.no_grad():
            # BERT
            bert_inputs = self.bert_tokenizer(text, return_tensors='pt', truncation=True,
                                             padding='max_length', max_length=128)
            bert_inputs = {k: v.to(self.device) for k, v in bert_inputs.items()}
            bert_feat = self.bert_model(**bert_inputs).last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            
            # RoBERTa
            roberta_inputs = self.roberta_tokenizer(text, return_tensors='pt', truncation=True,
                                                   padding='max_length', max_length=128)
            roberta_inputs = {k: v.to(self.device) for k, v in roberta_inputs.items()}
            roberta_feat = self.roberta_model(**roberta_inputs).last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            
            # CLIP
            clip_inputs = self.clip_processor(text=[text], return_tensors='pt', truncation=True,
                                             padding=True, max_length=77)
            clip_inputs = {k: v.to(self.device) for k, v in clip_inputs.items()}
            clip_feat = self.clip_model.get_text_features(**clip_inputs).squeeze().cpu().numpy()
        
        return np.concatenate([bert_feat, roberta_feat, clip_feat])  # 2048-dim

text_extractor = TextExtractor(device)

In [ ]:
# Extract text features from TRAIN data
# This will take ~1-1.5 hours for 75K texts

print("📝 Extracting text features from TRAINING data...")
print(f"   Processing {len(train_df)} texts...\n")

train_text_features = []

for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Train Text"):
    features = text_extractor.extract(str(row['catalog_content']))
    train_text_features.append(features)
    
    if idx % 500 == 0 and idx > 0:
        gc.collect()
        torch.cuda.empty_cache()

train_text_df = pd.DataFrame(train_text_features,
                             columns=[f'text_{i}' for i in range(2048)])

print(f"\n✓ Train text features extracted!")
print(f"   Shape: {train_text_df.shape}")

In [ ]:
# Extract text features from TEST data
# This will take ~1-1.5 hours for 75K texts

print("📝 Extracting text features from TEST data...")
print(f"   Processing {len(test_df)} texts...\n")

test_text_features = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test Text"):
    features = text_extractor.extract(str(row['catalog_content']))
    test_text_features.append(features)
    
    if idx % 500 == 0 and idx > 0:
        gc.collect()
        torch.cuda.empty_cache()

test_text_df = pd.DataFrame(test_text_features,
                            columns=[f'text_{i}' for i in range(2048)])

print(f"\n✓ Test text features extracted!")
print(f"   Shape: {test_text_df.shape}")

# Free memory
del text_extractor
gc.collect()
torch.cuda.empty_cache()

## Section 5: Advanced Feature Engineering

In [ ]:
# Create advanced engineered features

def create_advanced_features(df):
    print(f"Creating advanced features for {len(df)} samples...")
    features = pd.DataFrame()
    
    # Text statistics
    features['text_len'] = df['catalog_content'].str.len()
    features['word_count'] = df['catalog_content'].str.split().str.len()
    features['avg_word_len'] = features['text_len'] / (features['word_count'] + 1)
    features['upper_ratio'] = df['catalog_content'].str.findall(r'[A-Z]').str.len() / (features['text_len'] + 1)
    features['digit_count'] = df['catalog_content'].str.findall(r'\d').str.len()
    features['has_price'] = df['catalog_content'].str.contains(r'\$|price|cost', case=False).astype(int)
    
    # Brands
    brands = ['sony','samsung','apple','lg','hp','dell','lenovo','nike','adidas']
    for brand in brands:
        features[f'brand_{brand}'] = df['catalog_content'].str.lower().str.contains(brand).astype(int)
    
    # Categories
    categories = ['electronic','clothing','book','home','toy','sport','beauty','food']
    for cat in categories:
        features[f'cat_{cat}'] = df['catalog_content'].str.lower().str.contains(cat).astype(int)
    
    print(f"✓ Created {features.shape[1]} advanced features")
    return features

print("Creating features for train...")
train_advanced = create_advanced_features(train_df)

print("\nCreating features for test...")
test_advanced = create_advanced_features(test_df)

In [ ]:
# Combine all features

print("Combining all features...")

X_train = pd.concat([train_vision_df, train_text_df, train_advanced], axis=1)
X_test = pd.concat([test_vision_df, test_text_df, test_advanced], axis=1)
y_train = train_df['price'].values

# Fill NaN
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print(f"\n✓ Final feature matrix:")
print(f"   Train: {X_train.shape}")
print(f"   Test: {X_test.shape}")
print(f"\n   Vision: 4608 features")
print(f"   Text: 2048 features")
print(f"   Advanced: {train_advanced.shape[1]} features")
print(f"   TOTAL: {X_train.shape[1]} features 🔥")

## Section 6: Train Ensemble Models (6 Models)

**This will take 4-6 hours with 5-fold CV.**

In [ ]:
# Define SMAPE metric

def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

# Define 6 diverse models
print("Initializing 6 ML models...\n")

models = {
    'xgb_1': xgb.XGBRegressor(
        n_estimators=1500, learning_rate=0.03, max_depth=10,
        subsample=0.8, colsample_bytree=0.8,
        tree_method='gpu_hist', gpu_id=0, random_state=42
    ),
    'xgb_2': xgb.XGBRegressor(
        n_estimators=1200, learning_rate=0.05, max_depth=8,
        subsample=0.7, colsample_bytree=0.7,
        tree_method='gpu_hist', gpu_id=0, random_state=123
    ),
    'lgb_1': lgb.LGBMRegressor(
        n_estimators=1500, learning_rate=0.03, max_depth=10,
        subsample=0.8, colsample_bytree=0.8,
        device='gpu', random_state=42
    ),
    'lgb_2': lgb.LGBMRegressor(
        n_estimators=1200, learning_rate=0.05, max_depth=8,
        subsample=0.7, colsample_bytree=0.7,
        device='gpu', random_state=123
    ),
    'cat_1': CatBoostRegressor(
        iterations=1500, learning_rate=0.03, depth=10,
        task_type='GPU', verbose=False, random_state=42
    ),
    'cat_2': CatBoostRegressor(
        iterations=1200, learning_rate=0.05, depth=8,
        task_type='GPU', verbose=False, random_state=123
    )
}

print("✓ Models initialized!")
print("\nModels to train:")
for i, name in enumerate(models.keys(), 1):
    print(f"   {i}. {name}")

In [ ]:
# Train models with 5-fold CV
# This is the longest step: 4-6 hours

print("\n🤖 Training 6 models with 5-fold CV...")
print("   This will take 4-6 hours\n")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = {name: np.zeros(len(X_train)) for name in models.keys()}
test_preds = {name: np.zeros(len(X_test)) for name in models.keys()}
cv_scores = {name: [] for name in models.keys()}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/5")
    print(f"{'='*60}")
    
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
    for name, model in models.items():
        print(f"\n  Training {name}...")
        
        if 'xgb' in name:
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                     early_stopping_rounds=100, verbose=False)
        elif 'lgb' in name:
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                     callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
        else:
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val),
                     early_stopping_rounds=100, verbose=False)
        
        val_pred = model.predict(X_val)
        oof_preds[name][val_idx] = val_pred
        test_preds[name] += model.predict(X_test) / 5
        
        score = smape(y_val, val_pred)
        cv_scores[name].append(score)
        print(f"     SMAPE: {score:.4f}%")
    
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n\n{'='*60}")
print("TRAINING COMPLETE!")
print(f"{'='*60}")

In [ ]:
# Display individual model results

print("\n📊 INDIVIDUAL MODEL RESULTS:\n")
print(f"{'Model':<12} {'Mean CV':<12} {'OOF Score':<12}")
print("-" * 40)

for name in models.keys():
    mean_cv = np.mean(cv_scores[name])
    oof_score = smape(y_train, oof_preds[name])
    print(f"{name:<12} {mean_cv:>10.4f}%  {oof_score:>10.4f}%")

print("\n" + "="*60)

## Section 7: Meta-Stacking Layer

In [ ]:
# Train meta-model on top of base models

print("🔗 Training meta-stacking layer...\n")

# Create meta-features
meta_train = np.column_stack([oof_preds[name] for name in models.keys()])
meta_test = np.column_stack([test_preds[name] for name in models.keys()])

print(f"Meta-features shape: {meta_train.shape}")

# Train meta-model
meta_model = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.01, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='gpu_hist', random_state=42
)

meta_oof = np.zeros(len(meta_train))
meta_pred = np.zeros(len(meta_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(meta_train)):
    print(f"  Meta-fold {fold + 1}/5...")
    meta_model.fit(meta_train[tr_idx], y_train[tr_idx],
                  eval_set=[(meta_train[val_idx], y_train[val_idx])],
                  early_stopping_rounds=50, verbose=False)
    meta_oof[val_idx] = meta_model.predict(meta_train[val_idx])
    meta_pred += meta_model.predict(meta_test) / 5

meta_cv = smape(y_train, meta_oof)
print(f"\n✓ Meta-stacking CV: {meta_cv:.4f}%")

## Section 8: Create Multiple Ensemble Strategies

In [ ]:
# Create 5 different ensemble strategies

print("📊 Creating ensemble strategies...\n")

# 1. Simple average
simple_avg = sum(test_preds.values()) / len(models)
simple_oof = sum(oof_preds.values()) / len(models)

# 2. Weighted by CV score
weights = np.array([1.0 / np.mean(cv_scores[name]) for name in models.keys()])
weights = weights / weights.sum()
weighted_avg = sum(test_preds[name] * w for name, w in zip(models.keys(), weights))
weighted_oof = sum(oof_preds[name] * w for name, w in zip(models.keys(), weights))

# 3. Best 3 models
sorted_models = sorted(models.keys(), key=lambda x: np.mean(cv_scores[x]))
best_3_avg = sum(test_preds[name] for name in sorted_models[:3]) / 3
best_3_oof = sum(oof_preds[name] for name in sorted_models[:3]) / 3

# 4. Rank averaging
rank_preds = []
for name in models.keys():
    ranks = pd.Series(test_preds[name]).rank(pct=True)
    rank_preds.append(ranks.values)
rank_avg = np.mean(rank_preds, axis=0)
rank_avg_prices = np.percentile(y_train, rank_avg * 100)

# 5. Meta-stacking
meta_final = meta_pred

print("✓ Created 5 ensemble strategies!")

# Calculate scores
ensemble_scores = {
    'Simple Average': smape(y_train, simple_oof),
    'Weighted Average': smape(y_train, weighted_oof),
    'Best 3 Average': smape(y_train, best_3_oof),
    'Meta-Stacking': meta_cv
}

print("\n" + "="*60)
print("🏆 ENSEMBLE RESULTS")
print("="*60)

for name, score in ensemble_scores.items():
    print(f"{name:<20} {score:>10.4f}% SMAPE")

best_method = min(ensemble_scores, key=ensemble_scores.get)
best_score = ensemble_scores[best_method]

print("\n" + "="*60)
print(f"🥇 BEST METHOD: {best_method}")
print(f"🥇 BEST SCORE: {best_score:.4f}% CV")
print("="*60)

print(f"\n📊 Your Phase 5: 57.900%")
print(f"📊 Improvement: {57.900 - best_score:.2f}%")
print(f"📊 Expected LB: {best_score * 0.98:.2f}% - {best_score * 1.02:.2f}%")

if best_score < 46:
    print("\n🔥🔥🔥 AMAZING! TOP 10-30 POTENTIAL! 🔥🔥🔥")
elif best_score < 50:
    print("\n⭐⭐⭐ EXCELLENT! TOP 30-100 POTENTIAL! ⭐⭐⭐")
elif best_score < 54:
    print("\n✅✅✅ VERY GOOD! TOP 100-300 POTENTIAL! ✅✅✅")
else:
    print("\n✅ GOOD! Beats Phase 5! ✅")

## Section 9: Create Submissions

In [ ]:
# Create 5 submission files

print("💾 Creating submission files...\n")

submissions = {
    'simple_avg': simple_avg,
    'weighted_avg': weighted_avg,
    'best_3_avg': best_3_avg,
    'rank_avg': rank_avg_prices,
    'meta_stacking': meta_final
}

for name, preds in submissions.items():
    df = pd.DataFrame({
        'sample_id': test_df['sample_id'],
        'price': preds
    })
    filename = f'submission_{name}.csv'
    df.to_csv(filename, index=False)
    print(f"   ✓ {filename}")

print("\n" + "="*60)
print("✅ ALL SUBMISSIONS CREATED!")
print("="*60)

print("\n📋 SUBMISSION PRIORITY:")
print("   1️⃣ submission_meta_stacking.csv (BEST)")
print("   2️⃣ submission_best_3_avg.csv (Backup)")
print("   3️⃣ submission_weighted_avg.csv (Safe)")
print("   4️⃣ submission_simple_avg.csv (Conservative)")
print("   5️⃣ submission_rank_avg.csv (Experimental)")

print("\n🎯 Next steps:")
print("   1. Upload submission_meta_stacking.csv to Kaggle")
print("   2. Wait 5-10 minutes for scoring")
print("   3. Check your new rank!")
print("   4. Try other submissions if needed")

print("\n🔥 Good luck! 🔥")